# Digital Connect 4 Practice
Play against the AI directly in this notebook.

In [1]:
import sys
import os
import time
import numpy as np
from IPython.display import clear_output, display
from stable_baselines3 import PPO

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from pefforza.envs.connect4_env import Connect4Env
# Voice might optional in notebook
try:
    from pefforza.interaction.voice import VoiceEngine
    VOICE_AVAILABLE = True
except ImportError:
    VOICE_AVAILABLE = False
    print("Voice engine not available")


In [2]:
# Setup
env = Connect4Env(render_mode=None)
model_path = os.path.join(project_root, "pefforza/agent/models/notebook_model.zip")

model = None
if os.path.exists(model_path):
    try:
        model = PPO.load(model_path)
        print("Model loaded.")
    except Exception as e:
        print(f"Error loading model: {e}")
else:
    print("Model not found, playing against random agent.")


Model loaded.


In [3]:
def print_board(board):
    rows, cols = board.shape
    print("\n  1 2 3 4 5 6 7")
    print("  -------------")
    for r in range(rows):
        row_str = "|"
        for c in range(cols):
            val = board[r, c]
            if val == 0: char = " "
            elif val == 1: char = "X"
            elif val == 2: char = "O"
            row_str += f" {char}"
        row_str += " |"
        print(row_str)
    print("  -------------\n")

def run_game():
    obs, info = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        clear_output(wait=True)
        print("Player 1 (X) vs AI (O)")
        print_board(obs)
        
        if env.current_player == 1:
            # Human Turn
            valid = False
            while not valid:
                try:
                    move = input("Your Move (1-7): ")
                    if move.lower() == 'q': return
                    col = int(move) - 1
                    if 0 <= col <= 6:
                         # Check if full
                         if obs[0, col] == 0:
                            valid = True
                         else:
                            print("Column full!")
                    else:
                        print("Invalid column (1-7)")
                except ValueError:
                    print("Please enter a number.")
            
            obs, reward, terminated, truncated, info = env.step(col)
            
        else:
            # AI Turn
            print("AI Thinking...")
            time.sleep(0.5)
            if model:
                # Swap player IDs for model
                model_obs = obs.copy()
                model_obs[obs == 1] = 2
                model_obs[obs == 2] = 1
                action, _ = model.predict(model_obs, deterministic=True)
                action = int(action)
            else:
                action = env.action_space.sample()
            
            # Validity check
            if obs[0, action] != 0:
                valid_actions = [c for c in range(7) if obs[0, c] == 0]
                if valid_actions:
                    import random
                    action = random.choice(valid_actions)

            obs, reward, terminated, truncated, info = env.step(action)

    clear_output(wait=True)
    print_board(obs)
    if reward == 1.0:
        # Last player won. 
        # Env switches player after move. So if current is 2, Player 1 won.
        winner = "Player X" if env.current_player == 2 else "AI O"
        print(f"GAME OVER! {winner} Wins!")
    else:
        print("GAME OVER! Draw!")

run_game()


  1 2 3 4 5 6 7
  -------------
|               |
|               |
|       X       |
|   X X X       |
| O O O O       |
| X X X O O O   |
  -------------

GAME OVER! Player X Wins!
